# Hanys → 3D GLB (Colab GPU)
Wybierz Runtime → Change runtime type → T4 GPU przed Run all.

In [ ]:
import subprocess
p=subprocess.run(['nvidia-smi'],capture_output=True,text=True)
print(p.stdout)
if p.returncode!=0: raise RuntimeError('BRAK GPU. Wybierz Runtime → Change runtime type → T4 GPU i uruchom ponownie.')

In [ ]:
%cd /content
!rm -rf InstantMesh
!git clone --depth 1 https://github.com/TencentARC/InstantMesh.git
%cd /content/InstantMesh
!pip install -q -U pip setuptools wheel ninja
!pip install -q -r requirements.txt --no-build-isolation


In [ ]:
import torch
print('PyTorch:',torch.__version__)
print('CUDA:',torch.version.cuda)
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,1))

In [ ]:
import requests
url='https://raw.githubusercontent.com/bajerskim11-eng/beboki-katowice-mis/main/public/beboki/hanys.jpeg'
r=requests.get(url,timeout=30); r.raise_for_status()
open('/content/InstantMesh/hanys.jpeg','wb').write(r.content)
print('Hanys downloaded:',len(r.content),'bytes')

In [ ]:
%cd /content/InstantMesh
!mkdir -p outputs
!python run.py configs/instant-mesh-base.yaml hanys.jpeg --export_texmap


In [ ]:
import glob,os,trimesh
objs=glob.glob('/content/InstantMesh/outputs/**/*.obj',recursive=True)
print('OBJ:',objs)
if not objs: raise RuntimeError('Brak OBJ — sprawdź log poprzedniej komórki.')
mesh=trimesh.load(objs[-1],force='mesh')
out='/content/hanys.glb'; mesh.export(out)
print('GOTOWE:',out,os.path.getsize(out),'bytes')

In [ ]:
from google.colab import files
files.download('/content/hanys.glb')